# ⚙️ Data Processing

---
## Purpose

This notebook transforms the raw GRAZPEDWRI-DX dataset into a clean, training-ready YOLO-format dataset stored at `IronGear/data/yolo_dataset/`. This processed dataset is then used by all three sprints — no sprint should re-run data processing.

---

## What This Notebook Does

### Step 1 — Build Master Index
Joins the image file index, label file index, and patient metadata CSV into one unified table (`master_index.csv`). Every row = one image with its full path, label path, patient ID, and assigned split.

### Step 2 — Patient-Level Split
Splits 6,091 unique patients into train (70%), val (15%), test (15%) using stratified sampling. The stratification ensures each split has a similar ratio of positive (has annotation) to negative (no annotation) patients. **A patient's images will never appear in more than one split.**

### Step 3 — Label Remapping
Remaps the 9 raw class IDs from meta.yaml to the 5 Sprint 1 project class IDs:

| Raw Name | Raw ID | Project Name | Project ID |
|---|---|---|---|
| fracture | varies | fracture | 0 |
| metal | varies | metal_implant | 1 |
| periostealreaction | varies | periosteal_reaction | 2 |
| pronatorsign | varies | pronator_sign | 3 |
| softtissue | varies | soft_tissue | 4 |
| boneanomaly, bonelesion, foreignbody, text | — | (ignored) | — |

### Step 4 — Oversampling
Rare class images in the **train split only** are duplicated to reduce class imbalance:

| Class | Copies Added | Rationale |
|---|---|---|
| metal_implant | ×5 | ~820 annotations — 22× fewer than fracture |
| periosteal_reaction | ×2 | ~3,450 annotations — moderate imbalance |
| pronator_sign | ×5 | ~570 annotations — very rare |
| soft_tissue | ×6 | ~460 annotations — rarest class |

### Step 5 — YOLO Dataset Structure
Writes the final dataset in the standard YOLO folder layout:

```
IronGear/data/yolo_dataset/
├── dataset.yaml          ← YOLO config (path, classes, nc)
├── images/
│   ├── train/            ← images (incl. oversampled copies)
│   ├── val/              ← original val images only
│   └── test/             ← original test images only
└── labels/
    ├── train/            ← remapped .txt label files
    ├── val/
    └── test/
```

---

## Restart Safety

> ✅ **Re-running is safe at any time.**
> Marker files (`.dataset_built`, `.oversampled`) prevent duplicate processing.
> After a kernel restart, run **Cell 2 (Paths & Config)** first.

---

## Execution Order

```
Cell 1  →  Install dependencies
Cell 2  →  Paths & config              ⚠️ Always run after restart
Cell 3  →  Read class mapping from meta.yaml
Cell 4  →  Scan images + labels, build master index  (skips if CSV exists)
Cell 5  →  Patient-level split          (skips if split already assigned)
Cell 6  →  Build YOLO dataset           (skips if .dataset_built marker exists)
Cell 7  →  Oversample rare classes      (skips if .oversampled marker exists)
Cell 8  →  Validate processed dataset
Cell 9  →  Write dataset.yaml
Cell 10 →  Processing figures and reports
Cell 11 →  Save processing report
```


---
## 1 — Install Dependencies


In [1]:
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "pandas", "scikit-learn", "matplotlib", "tqdm", "pyyaml"],
    check=True
)
print("✓ Dependencies ready.")

✓ Dependencies ready.


---
## 2 — Paths & Config ⚠️

> **Always run this cell first after any kernel restart.**

### Key Variables Defined Here

| Variable | Description |
|---|---|
| `YOLO_DIR` | Output root for the YOLO dataset |
| `RAW_TO_PROJECT` | Maps raw meta.yaml class names → clean project names |
| `PROJECT_CLASSES` | Ordered list of 5 target class names (index = class ID) |
| `OVERSAMPLE` | Dict of class → number of extra copies to add to train split |
| `TRAIN_RATIO`, `VAL_RATIO` | Split proportions (test = 1 - train - val) |


In [14]:
from pathlib import Path
import json

# ─── EFS Auto-Detection ───────────────────────────────────────────────────────
_CANDIDATES = [
    Path("/home/sagemaker-user/user-default-efs"),
    Path("/home/sagemaker-user"),
    Path.home() / "user-default-efs",
    Path.home(),
]
EFS      = next((c for c in _CANDIDATES if (c / "IronGear").exists()), _CANDIDATES[0])
IRONGEAR = EFS / "IronGear"
DATA_DIR = IRONGEAR / "data"

# ─── Load Path Config Saved by Notebook 00 ───────────────────────────────────
# This JSON was saved by Notebook 00 so we don't need to re-discover paths
path_cfg_file = DATA_DIR / "reports" / "dataset_paths.json"
assert path_cfg_file.exists(), (
    f"dataset_paths.json not found.\n"
    f"Run Notebook 00 first.\nExpected: {path_cfg_file}"
)
with open(path_cfg_file) as f:
    pc = json.load(f)

# Reconstruct Path objects from the saved string paths
DATASET_CSV = Path(pc["dataset_csv"])
META_YAML   = Path(pc["meta_yaml"])
LABELS_DIR  = Path(pc["labels_dir"])
IMAGE_DIRS  = [Path(d) for d in pc["image_dirs"]]

# ─── Output Paths ─────────────────────────────────────────────────────────────
YOLO_DIR    = DATA_DIR / "yolo_dataset"     # final YOLO dataset root
REPORTS_DIR = DATA_DIR / "reports"          # JSON reports and master index CSV
FIG_DIR     = DATA_DIR / "figures" / "processing"  # figures from this notebook

for d in [YOLO_DIR, REPORTS_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ─── Split Configuration ──────────────────────────────────────────────────────
TRAIN_RATIO = 0.70   # 70% of patients go to training
VAL_RATIO   = 0.15   # 15% to validation; remaining 15% to test
SEED        = 42     # fixed random seed for reproducibility

# ─── Sprint 1: 5 Target Classes ───────────────────────────────────────────────
# Maps EXACT lowercase raw names from meta.yaml to clean project names.
# IMPORTANT: these keys must match meta.yaml exactly (e.g. 'periostealreaction' not 'periosteal_reaction')
RAW_TO_PROJECT = {
    "fracture"          : "fracture",
    "metal"             : "metal_implant",
    "periostealreaction": "periosteal_reaction",
    "pronatorsign"      : "pronator_sign",
    "softtissue"        : "soft_tissue",
    # All other classes (boneanomaly, bonelesion, foreignbody, text) are ignored in Sprint 1
}

# The ordered list defines the final class IDs: index 0 = fracture, index 1 = metal_implant, etc.
PROJECT_CLASSES = [
    "fracture",            # class ID 0
    "metal_implant",       # class ID 1
    "periosteal_reaction", # class ID 2
    "pronator_sign",       # class ID 3
    "soft_tissue",         # class ID 4
]
CLASS_TO_ID = {name: idx for idx, name in enumerate(PROJECT_CLASSES)}

# ─── Oversampling Config ──────────────────────────────────────────────────────
# Applied to TRAIN split only. n_copies = number of extra duplicate copies to add.
# Higher multiplier = rarer class needs more balancing.
OVERSAMPLE = {
    "metal_implant"       : 5,   # ~820 annotations → very rare
    "periosteal_reaction" : 2,   # ~3,450 annotations → moderately rare
    "pronator_sign"       : 5,   # ~570 annotations → very rare
    "soft_tissue"         : 6,   # ~460 annotations → rarest class
}

print("✓ Config loaded")
print(f"  EFS base   : {EFS}")
print(f"  Data dir   : {DATA_DIR}")
print(f"  YOLO dir   : {YOLO_DIR}")
print(f"  Split      : {TRAIN_RATIO:.0%} train / {VAL_RATIO:.0%} val / {1-TRAIN_RATIO-VAL_RATIO:.0%} test")
print(f"  Classes    : {PROJECT_CLASSES}")

✓ Config loaded
  EFS base   : /home/sagemaker-user/user-default-efs
  Data dir   : /home/sagemaker-user/user-default-efs/IronGear/data
  YOLO dir   : /home/sagemaker-user/user-default-efs/IronGear/data/yolo_dataset
  Split      : 70% train / 15% val / 15% test
  Classes    : ['fracture', 'metal_implant', 'periosteal_reaction', 'pronator_sign', 'soft_tissue']


---
## 3 — Read Class Mapping from meta.yaml

Loads the raw class IDs from `meta.yaml` and builds a direct mapping of **raw class ID → project class ID**. This mapping is used in Cell 6 when processing every label file.

The assertion at the end verifies that exactly 5 raw classes were successfully mapped. If this fails, check that the keys in `RAW_TO_PROJECT` (Cell 2) match the actual class names printed by this cell.


In [3]:
import yaml

# ─── Load meta.yaml ───────────────────────────────────────────────────────────
with open(META_YAML) as f:
    meta = yaml.safe_load(f)

names = meta.get("names", {})
# Handle both list and dict format
RAW_CLASS_MAP = (
    {i: n for i, n in enumerate(names)}
    if isinstance(names, list)
    else {int(k): v for k, v in names.items()}
)   # result: {raw_int_id: "raw_name_string"}

# ─── Build Raw ID → Project ID Mapping ────────────────────────────────────────
# For each raw class, look up its name in RAW_TO_PROJECT.
# If found, map raw_id → project_id (the index in PROJECT_CLASSES list)
RAW_ID_TO_PROJ_ID = {}   # {raw_int_id: project_int_id}

print(f"{'Raw ID':>6}  {'Raw Name':<25}  {'→  Project Name':<28}  Project ID")
print("─" * 72)

for raw_id, raw_name in sorted(RAW_CLASS_MAP.items()):
    # Normalise the raw name to lowercase for matching
    proj_name = RAW_TO_PROJECT.get(raw_name.lower().strip())

    if proj_name is not None:
        proj_id = CLASS_TO_ID[proj_name]          # look up the project class integer ID
        RAW_ID_TO_PROJ_ID[raw_id] = proj_id       # store the mapping
        print(f"{raw_id:>6}  {raw_name:<25}  →  {proj_name:<28}  {proj_id}")
    else:
        # This class is not one of our 5 targets — will be dropped during label processing
        print(f"{raw_id:>6}  {raw_name:<25}  →  {'(ignored)':<28}  –")

# ─── Validation ───────────────────────────────────────────────────────────────
assert len(RAW_ID_TO_PROJ_ID) == 5, (
    f"Expected exactly 5 mapped classes, got {len(RAW_ID_TO_PROJ_ID)}.\n"
    f"Check that RAW_TO_PROJECT keys in Cell 2 match the raw names printed above.\n"
    f"Mapped so far: {RAW_ID_TO_PROJ_ID}"
)

print(f"\n✓ All 5 target classes mapped correctly")

Raw ID  Raw Name                   →  Project Name               Project ID
────────────────────────────────────────────────────────────────────────
     0  boneanomaly                →  (ignored)                     –
     1  bonelesion                 →  (ignored)                     –
     2  foreignbody                →  (ignored)                     –
     3  fracture                   →  fracture                      0
     4  metal                      →  metal_implant                 1
     5  periostealreaction         →  periosteal_reaction           2
     6  pronatorsign               →  pronator_sign                 3
     7  softtissue                 →  soft_tissue                   4
     8  text                       →  (ignored)                     –

✓ All 5 target classes mapped correctly


---
## 4 — Scan Images + Labels, Build Master Index

Scans all image folders and label files, then joins them with patient metadata from `dataset.csv` to build a single unified `master_index.csv`. Each row in this table represents one image with its associated metadata.

### Master Index Columns

| Column | Description |
|---|---|
| `stem` | Image filename without extension |
| `image_path` | Full absolute path to the image file |
| `ext` | File extension (`.png`, `.jpg`, etc.) |
| `patient_id` | Unique patient identifier from dataset.csv |
| `label_path` | Full path to the YOLO .txt label file |
| `has_label` | True if a label file exists for this image |
| `split` | train / val / test (added in Cell 5) |

### Restart Safety
If `master_index.csv` already exists on disk, this cell loads it instead of re-scanning — so re-running after a restart is fast.


In [4]:
import pandas as pd
from tqdm.notebook import tqdm

IMG_EXTS    = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
master_csv  = REPORTS_DIR / "master_index.csv"    # output CSV path

# ─── Skip if Already Built ────────────────────────────────────────────────────
if master_csv.exists():
    master = pd.read_csv(master_csv)
    print(f"✓ master_index.csv loaded from disk ({len(master):,} rows)")
    print(f"  (Delete {master_csv} to rebuild from scratch)")
else:
    # ─── Scan All Image Folders ───────────────────────────────────────────────
    img_records = []
    for folder in IMAGE_DIRS:
        for p in sorted(folder.rglob("*")):
            if p.is_file() and p.suffix.lower() in IMG_EXTS:
                img_records.append({
                    "stem"      : p.stem,
                    "image_path": str(p),
                    "ext"       : p.suffix.lower(),
                })
    img_df = pd.DataFrame(img_records).drop_duplicates("stem", keep="first")
    print(f"  Images scanned : {len(img_df):,}")

    # ─── Scan Label Folder ────────────────────────────────────────────────────
    lbl_records = []
    for p in sorted(LABELS_DIR.rglob("*.txt")):
        if p.is_file():
            lbl_records.append({"stem": p.stem, "label_path": str(p)})
    lbl_df = pd.DataFrame(lbl_records).drop_duplicates("stem", keep="first")
    print(f"  Labels scanned : {len(lbl_df):,}")

    # ─── Load dataset.csv and Extract Patient IDs ─────────────────────────────
    meta_df     = pd.read_csv(DATASET_CSV)
    cols_lower  = {c.lower().strip(): c for c in meta_df.columns}

    # Find the patient_id and filestem columns (handle varying capitalisation)
    patient_col  = cols_lower.get("patient_id") or cols_lower.get("patientid")
    filestem_col = cols_lower.get("filestem") or cols_lower.get("file_name") or cols_lower.get("filename")

    assert patient_col,  f"patient_id column not found. Available: {list(meta_df.columns)}"
    assert filestem_col, f"filestem column not found. Available: {list(meta_df.columns)}"

    # Prepare a slim metadata table with just stem and patient_id for the join
    meta_slim = meta_df[[filestem_col, patient_col]].copy()
    meta_slim["stem"] = meta_slim[filestem_col].astype(str).apply(lambda x: Path(x.strip()).stem)
    meta_slim = meta_slim[["stem", patient_col]].rename(columns={patient_col: "patient_id"})

    # ─── Merge All Three Tables ───────────────────────────────────────────────
    # Left join on stem so every image row is kept even if metadata is missing
    master = img_df.merge(meta_slim, on="stem", how="left")
    master = master.merge(lbl_df,    on="stem", how="left")
    master["has_label"] = master["label_path"].notna()  # True if label file exists

    # ─── Fill Missing Patient IDs ─────────────────────────────────────────────
    # Fallback: derive patient ID from the numeric prefix of the filename
    # e.g. '0003_0662359226_01_WRI-R1_M011' → patient_id = 3
    n_missing = master["patient_id"].isna().sum()
    if n_missing > 0:
        print(f"  Warning: {n_missing:,} images have no patient_id match — using filename prefix.")
        master["patient_id"] = master["patient_id"].fillna(
            master["stem"].apply(
                lambda s: int(s.split("_")[0]) if s.split("_")[0].isdigit() else s
            )
        )

    # ─── Save to Disk ─────────────────────────────────────────────────────────
    master.to_csv(master_csv, index=False)
    print(f"  Master index saved: {master_csv}")

print(f"\nMaster index summary:")
print(f"  Total rows          : {len(master):,}")
print(f"  Has label file      : {master['has_label'].sum():,}")
print(f"  Unique patients     : {master['patient_id'].nunique():,}")
master.head(3)

  Images scanned : 20,327
  Labels scanned : 20,327
  Master index saved: /home/sagemaker-user/user-default-efs/IronGear/data/reports/master_index.csv

Master index summary:
  Total rows          : 20,327
  Has label file      : 20,327
  Unique patients     : 6,091


,stem,image_path,ext,patient_id,label_path,has_label
0,0001_1297860395_01_WRI-L1_M014,/home/sagemaker-user/user-default-efs/IronGear...,.png,1,/home/sagemaker-user/user-default-efs/IronGear...,True
1,0001_1297860435_01_WRI-L2_M014,/home/sagemaker-user/user-default-efs/IronGear...,.png,1,/home/sagemaker-user/user-default-efs/IronGear...,True
2,0002_0354485735_01_WRI-R1_F012,/home/sagemaker-user/user-default-efs/IronGear...,.png,2,/home/sagemaker-user/user-default-efs/IronGear...,True


---
## 5 — Patient-Level Train / Val / Test Split

Assigns every patient to exactly one of train, val, or test. The split is done at the **patient level** (not image level) to prevent data leakage.

### Algorithm
1. Build a one-row-per-patient summary with a `has_positive` flag (does this patient have at least one annotated image?)
2. Use `train_test_split` with `stratify=has_positive` to ensure similar positive/negative ratios in each split
3. Map the patient→split assignment back to every image row in the master index
4. Run a leakage check: assert no patient ID appears in more than one split

### Leakage Check
If this check fails, **stop immediately** — it means the same patient would be in training and evaluation, making test metrics meaningless.


In [6]:
import numpy as np
from sklearn.model_selection import train_test_split

# ─── Skip if Split Already Assigned ─────────────────────────────────────────
if "split" in master.columns and master["split"].notna().all():
    print("✓ Split already assigned in master_index.csv — skipping.")
else:
    # ─── One Row Per Patient ──────────────────────────────────────────────────
    # Aggregate: a patient 'has_positive' if ANY of their images has a label file
    patients = (
        master[["patient_id", "has_label"]]
        .groupby("patient_id")["has_label"].any()
        .reset_index()
        .rename(columns={"has_label": "has_positive"})
    )
    N = len(patients)
    print(f"  Unique patients to split: {N:,}")

    # ─── Stratified Split ─────────────────────────────────────────────────────
    # Attempt stratified split on has_positive flag (ensures similar +/- ratio)
    # Falls back to random split if stratification is numerically unstable
    try:
        # First split: 70% train, 30% temp (to be split into val+test)
        train_p, temp_p = train_test_split(
            patients["patient_id"].values,
            test_size   = 1 - TRAIN_RATIO,
            random_state= SEED,
            stratify    = patients["has_positive"].values,
        )
        # Get the has_positive values for the temp group for second stratification
        temp_strat = patients.loc[
            patients["patient_id"].isin(temp_p), "has_positive"].values

        # Second split: 50/50 of temp → gives 15% val and 15% test
        val_p, test_p = train_test_split(
            temp_p,
            test_size    = 0.50,
            random_state = SEED,
            # Only stratify if both classes are present in temp group
            stratify     = temp_strat if len(set(temp_strat)) > 1 else None,
        )
        method = "stratified"

    except Exception as e:
        # Fallback: plain random split using numpy
        print(f"  Stratified split failed ({e}) — using random split.")
        rng = np.random.default_rng(SEED)
        ids = rng.permutation(patients["patient_id"].values)
        nt  = int(N * TRAIN_RATIO)
        nv  = int(N * VAL_RATIO)
        train_p = ids[:nt]; val_p = ids[nt:nt+nv]; test_p = ids[nt+nv:]
        method  = "random"

    # ─── Map Split Assignments Back to Image Level ───────────────────────────
    split_map = {pid: "train" for pid in train_p}
    split_map.update({pid: "val"  for pid in val_p})
    split_map.update({pid: "test" for pid in test_p})
    master["split"] = master["patient_id"].map(split_map).fillna("train")

    # ─── Leakage Check ────────────────────────────────────────────────────────
    # Verify zero overlap in patient IDs between any two splits
    for s1, s2 in [("train","val"),("train","test"),("val","test")]:
        overlap = (
            set(master.loc[master.split == s1, "patient_id"]) &
            set(master.loc[master.split == s2, "patient_id"])
        )
        assert not overlap, (
            f"PATIENT LEAKAGE between {s1}/{s2}: {len(overlap)} shared patients.\n"
            f"This should never happen — investigate before proceeding."
        )

    # Save the updated master index with split column
    master.to_csv(master_csv, index=False)
    print(f"  Split method: {method}")
    print(f"  ✓ Leakage check PASSED — no patient appears in more than one split")

# ─── Summary ──────────────────────────────────────────────────────────────────
print("\nImages per split:")
print(master["split"].value_counts().sort_index().to_string())
print("\nPatients per split:")
print(master.groupby("split")["patient_id"].nunique().sort_index().to_string())

  Unique patients to split: 6,091
  Split method: stratified
  ✓ Leakage check PASSED — no patient appears in more than one split

Images per split:
split
test      3115
train    14243
val       2969

Patients per split:
split
test      914
train    4263
val       914


---
## 6 — Build YOLO Dataset

Processes every image in the master index:
1. Reads the original YOLO label file
2. Remaps raw class IDs to project class IDs (dropping non-target classes)
3. Writes the new label file to the appropriate split folder
4. Copies the image to the matching split folder
5. Tracks rare-class train images for oversampling in Cell 7

### Negative Samples
Images with no target-class annotations get an **empty `.txt` label file**. This is required for YOLO training — without negatives the model learns to detect findings everywhere and produces excessive false positives.

> ⏱️ **Expected runtime:** 10–30 minutes (copies 20k images)
> A `.dataset_built` marker prevents re-running if already done.


In [8]:
import shutil
from collections import defaultdict

# ─── Helper Functions ─────────────────────────────────────────────────────────
def read_yolo(path: str) -> list:
    """Parse a YOLO label file. Returns list of (class_id, cx, cy, w, h) tuples."""
    rows = []
    try:
        with open(path) as f:
            for line in f:
                p = line.strip().split()
                if len(p) == 5:   # valid YOLO annotation line
                    rows.append((int(p[0]), float(p[1]),
                                  float(p[2]), float(p[3]), float(p[4])))
    except Exception:
        pass   # return empty list if file unreadable
    return rows

def write_yolo(rows: list, out_path: Path):
    """Write a list of (class_id, cx, cy, w, h) tuples to a YOLO .txt file."""
    with open(out_path, "w") as f:
        for (c, cx, cy, w, h) in rows:
            # 6 decimal places for bounding box precision
            f.write(f"{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")

# ─── Skip if Already Built ────────────────────────────────────────────────────
build_marker = YOLO_DIR / ".dataset_built"
if build_marker.exists():
    print(f"✓ YOLO dataset already built.")
    print(f"  (Delete {build_marker} to rebuild)")
else:
    # Create the 6 required split subdirectories
    for split in ["train", "val", "test"]:
        (YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

    # Counters and collectors
    stats = {
        "copied"  : 0,    # images successfully copied
        "negatives": 0,   # images written with empty label (no target class found)
        "kept"    : 0,    # annotation boxes kept (matched a target class)
        "dropped" : 0,    # annotation boxes dropped (non-target class)
        "per_class": defaultdict(int),  # per project class annotation count
    }
    # rare_images: stores (img_path, lbl_path, remapped_rows, ext) for train images
    # containing rare classes — used by Cell 7 for oversampling
    rare_images = defaultdict(list)   # {project_class_name: [(...), ...]}

    import pandas as pd
    print("Building YOLO dataset...")
    for _, row in tqdm(master.iterrows(), total=len(master), desc="Processing images"):
        split    = row["split"]
        stem     = row["stem"]
        img_path = Path(row["image_path"])
        lbl_path = row["label_path"]

        # ─── Remap Labels ──────────────────────────────────────────────────────
        remapped = []
        if row["has_label"] and pd.notna(lbl_path):
            for (raw_id, cx, cy, w, h) in read_yolo(lbl_path):
                proj_id = RAW_ID_TO_PROJ_ID.get(raw_id)   # None if not a target class
                if proj_id is not None:
                    remapped.append((proj_id, cx, cy, w, h))
                    stats["kept"] += 1
                    stats["per_class"][PROJECT_CLASSES[proj_id]] += 1
                else:
                    stats["dropped"] += 1   # non-target class — discard this box

        # If no target-class annotations → negative image (empty label file)
        if not remapped:
            stats["negatives"] += 1

        # ─── Write Remapped Label File ─────────────────────────────────────────
        # Always write a label file (even if empty) — YOLO requires 1:1 image:label
        write_yolo(remapped, YOLO_DIR / "labels" / split / f"{stem}.txt")

        # ─── Copy Image ────────────────────────────────────────────────────────
        if img_path.exists():
            shutil.copy2(img_path, YOLO_DIR / "images" / split / f"{stem}{row['ext']}")
            stats["copied"] += 1

        # ─── Track Rare Train Images for Oversampling ──────────────────────────
        # Only track training images that contain at least one rare class
        if split == "train" and remapped:
            cls_in_img = {PROJECT_CLASSES[r[0]] for r in remapped}
            for cls_name in cls_in_img:
                if cls_name in OVERSAMPLE:
                    rare_images[cls_name].append(
                        (img_path, lbl_path, remapped, row["ext"])
                    )

    build_marker.touch()  # mark as done
    print(f"\n✓ Dataset built")
    print(f"  Images copied      : {stats['copied']:,}")
    print(f"  Annotations kept   : {stats['kept']:,}")
    print(f"  Dropped (ignored)  : {stats['dropped']:,}")
    print(f"  Negative images    : {stats['negatives']:,}")
    print("\n  Annotations per project class:")
    for cls in PROJECT_CLASSES:
        print(f"    {cls:<25}: {stats['per_class'][cls]:>7,}")

# Show current split sizes
for split in ["train", "val", "test"]:
    n = len(list((YOLO_DIR / "images" / split).glob("*")))
    print(f"  {split}: {n:,} images")

Building YOLO dataset...


Processing images:   0%|          | 0/20327 [00:00<?, ?it/s]


✓ Dataset built
  Images copied      : 20,327
  Annotations kept   : 23,392
  Dropped (ignored)  : 24,051
  Negative images    : 6,495

  Annotations per project class:
    fracture                 :  18,090
    metal_implant            :     818
    periosteal_reaction      :   3,453
    pronator_sign            :     567
    soft_tissue              :     464
  train: 14,243 images
  val: 2,969 images
  test: 3,115 images


---
## 7 — Oversample Rare Classes (Train Only)

Creates duplicate copies of training images that contain rare classes. This is applied **only to the train split** — val and test remain the original unmodified images for fair evaluation.

### Naming Convention
Each oversampled copy gets a unique stem to avoid file conflicts:
```
original:  0003_0662359226_01_WRI-R1_M011.png
copy 0:    0003_0662359226_01_WRI-R1_M011_os1_0.png   (os = oversample, 1 = class ID)
copy 1:    0003_0662359226_01_WRI-R1_M011_os1_1.png
```

### Marker File
`.oversampled` prevents re-running. Delete it to redo oversampling with different multipliers.


In [9]:
# ─── Skip if Already Done ─────────────────────────────────────────────────────
os_marker = YOLO_DIR / ".oversampled"
if os_marker.exists():
    print("✓ Oversampling already done.")
    print(f"  (Delete {os_marker} to redo with different multipliers)")
else:
    os_added = defaultdict(int)   # track how many copies were added per class

    print("Oversampling rare classes in train split...")
    for cls_name, n_copies in OVERSAMPLE.items():
        cls_id = CLASS_TO_ID[cls_name]  # numeric class ID used in the filename

        for (img_path, lbl_path, remapped, ext) in rare_images.get(cls_name, []):
            original_stem = Path(img_path).stem

            # Create n_copies duplicates with unique stems
            for k in range(n_copies):
                new_stem = f"{original_stem}_os{cls_id}_{k}"   # e.g. 0003_..._os1_2
                out_img  = YOLO_DIR / "images" / "train" / f"{new_stem}{ext}"
                out_lbl  = YOLO_DIR / "labels" / "train" / f"{new_stem}.txt"

                # Only copy if the source image exists and the destination doesn't yet
                if img_path.exists() and not out_img.exists():
                    shutil.copy2(img_path, out_img)      # copy image
                    write_yolo(remapped, out_lbl)         # copy remapped labels
                    os_added[cls_name] += 1

    os_marker.touch()   # mark as done
    print("\n✓ Oversampling complete")
    for cls_name, n in os_added.items():
        print(f"  {cls_name:<25}: +{n:,} copies added")

# ─── Show Final Train Size ─────────────────────────────────────────────────────
final_train = len(list((YOLO_DIR / "images" / "train").glob("*")))
base_train  = master[master.split == "train"].shape[0]
print(f"\nTrain split: {base_train:,} base images → {final_train:,} after oversampling")
print(f"  ({final_train - base_train:,} oversampled copies added)")

Oversampling rare classes in train split...

✓ Oversampling complete
  metal_implant            : +2,375 copies added
  periosteal_reaction      : +3,090 copies added
  pronator_sign            : +1,910 copies added
  soft_tissue              : +1,836 copies added

Train split: 14,243 base images → 23,454 after oversampling
  (9,211 oversampled copies added)


---
## 8 — Validate Processed Dataset

Runs a series of checks on the processed YOLO dataset to confirm integrity before training:

1. **Image/label count parity** — every image folder should have a matching label folder count
2. **Missing labels** — no image should be without a `.txt` label file (even empty)
3. **Class ID bounds** — all class IDs in label files must be 0–4 (our 5 project classes)

If all checks pass, it is safe to proceed to training.


In [10]:
print("Validating processed YOLO dataset...")
print()
issues = []   # accumulate any problems found

print(f"{'Split':<8} {'Images':>8} {'Labels':>8} {'Positive':>10} {'Negative':>10}")
print("─" * 50)

for split in ["train", "val", "test"]:
    img_dir = YOLO_DIR / "images" / split
    lbl_dir = YOLO_DIR / "labels" / split

    # Get stems (filenames without extension) for images and labels
    img_stems = {p.stem for p in img_dir.glob("*")}
    lbl_stems = {p.stem for p in lbl_dir.glob("*.txt")}

    # Count positive (non-empty) and negative (empty) label files
    n_pos = sum(1 for p in lbl_dir.glob("*.txt") if p.stat().st_size > 0)
    n_neg = len(lbl_stems) - n_pos

    print(f"{split:<8} {len(img_stems):>8,} {len(lbl_stems):>8,} {n_pos:>10,} {n_neg:>10,}")

    # Check 1: every image has a corresponding label file
    missing_labels = img_stems - lbl_stems
    if missing_labels:
        issues.append(f"{split}: {len(missing_labels)} images missing label files")

    # Check 2: spot-check class IDs — all must be in 0–4
    # (Check first 1000 label files per split for speed)
    for lbl_path in list(lbl_dir.glob("*.txt"))[:1000]:
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cid = int(parts[0])
                    if cid not in range(len(PROJECT_CLASSES)):
                        issues.append(
                            f"{split}: Invalid class ID {cid} in {lbl_path.name}"
                        )

print()
if issues:
    print("Issues found:")
    for issue in issues:
        print(f"  ✗ {issue}")
    print("\nPlease resolve these issues before training.")
else:
    print("✓ All validation checks PASSED — dataset is ready for training.")

Validating processed YOLO dataset...

Split      Images   Labels   Positive   Negative
──────────────────────────────────────────────────
train      23,454   23,454     18,897      4,557
val         2,969    2,969      2,011        958
test        3,115    3,115      2,135        980

✓ All validation checks PASSED — dataset is ready for training.


---
## 9 — Write dataset.yaml

Writes the YOLO-format `dataset.yaml` configuration file that Ultralytics uses during training to locate images, labels and class names.

### Why We Re-Write It on Each Run
SageMaker EFS can be mounted at different absolute paths depending on the instance. By writing `dataset.yaml` dynamically using the current `YOLO_DIR` value, we always get the correct path regardless of which instance runs the training.


In [11]:
# Build the YAML content as a string.
# Using YOLO_DIR (direct EFS path) avoids symlink resolution issues
# that can cause Ultralytics to fail when the resolved path differs from the EFS path.
yaml_text = f"""# Iron Gear — GRAZPEDWRI-DX 5-class detection dataset
# Auto-generated by IronGear/01_data_processing.ipynb
# Reused by Sprint 1, Sprint 2, and Sprint 3

path: {YOLO_DIR}   # absolute path to dataset root
train: images/train
val:   images/val
test:  images/test

nc: 5   # number of classes

# Class names — index position = class ID used in label files
names:
  0: fracture
  1: metal_implant
  2: periosteal_reaction
  3: pronator_sign
  4: soft_tissue
"""

DATASET_YAML = YOLO_DIR / "dataset.yaml"
DATASET_YAML.write_text(yaml_text)

print(f"✓ dataset.yaml written to: {DATASET_YAML}")
print()
print(yaml_text)

✓ dataset.yaml written to: /home/sagemaker-user/user-default-efs/IronGear/data/yolo_dataset/dataset.yaml

# Iron Gear — GRAZPEDWRI-DX 5-class detection dataset
# Auto-generated by IronGear/01_data_processing.ipynb
# Reused by Sprint 1, Sprint 2, and Sprint 3

path: /home/sagemaker-user/user-default-efs/IronGear/data/yolo_dataset   # absolute path to dataset root
train: images/train
val:   images/val
test:  images/test

nc: 5   # number of classes

# Class names — index position = class ID used in label files
names:
  0: fracture
  1: metal_implant
  2: periosteal_reaction
  3: pronator_sign
  4: soft_tissue



---
## 10 — Processing Figures

Generates a summary visualisation showing the results of data processing — split sizes before and after oversampling, per-class annotation counts, and patient distribution across splits.


In [12]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Get final split image counts
total_train = len(list((YOLO_DIR/"images"/"train").glob("*")))
total_val   = len(list((YOLO_DIR/"images"/"val").glob("*")))
total_test  = len(list((YOLO_DIR/"images"/"test").glob("*")))
base_train  = int(master[master.split == "train"].shape[0])

COLORS_5 = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12", "#9b59b6"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Iron Gear — Data Processing Summary",
             fontsize=13, fontweight="bold")

# ─── Plot 1: Before vs After Oversampling in Train ────────────────────────────
ax = axes[0]
splits       = ["train", "val", "test"]
base_vals    = [base_train, total_val, total_test]   # counts before oversampling
final_vals   = [total_train, total_val, total_test]  # counts after oversampling
x = range(3)
ax.bar([i - 0.2 for i in x], base_vals,  0.38, label="Base",  color="#95a5a6")
ax.bar([i + 0.2 for i in x], final_vals, 0.38, label="After Oversampling", color="#2ecc71")
ax.set_xticks(list(x))
ax.set_xticklabels(splits, fontsize=11)
ax.set_title("Train Set: Before vs After Oversampling", fontweight="bold")
ax.set_ylabel("Image Count")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
for i, v in enumerate(final_vals):
    ax.text(i + 0.2, v + 30, f"{v:,}", ha="center", fontsize=8)

# ─── Plot 2: Annotations per Project Class ────────────────────────────────────
ax = axes[1]
cls_vals = [stats["per_class"].get(c, 0) for c in PROJECT_CLASSES]
bars = ax.barh(PROJECT_CLASSES, cls_vals, color=COLORS_5, edgecolor="white")
ax.set_title("Annotations per Project Class", fontweight="bold")
ax.set_xlabel("Annotation Count")
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.3)
for bar, v in zip(bars, cls_vals):
    ax.text(v + 20, bar.get_y() + bar.get_height() / 2,
            f"{v:,}", va="center", fontsize=8)

# ─── Plot 3: Patient Distribution Across Splits ───────────────────────────────
ax = axes[2]
pat_counts = master.groupby("split")["patient_id"].nunique()
ax.pie(
    [pat_counts.get(s, 0) for s in splits],
    labels=[f"{s}\n{pat_counts.get(s,0):,} patients" for s in splits],
    colors=["#3498db", "#e67e22", "#2ecc71"],
    autopct="%1.1f%%", startangle=90,
    textprops={"fontsize": 9},
    wedgeprops={"edgecolor": "white", "linewidth": 1.5},
)
ax.set_title("Patient Split Distribution", fontweight="bold")

plt.tight_layout()

# Save and display
fig_path = FIG_DIR / "01_processing_summary.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Figure saved: {fig_path}")

✓ Figure saved: /home/sagemaker-user/user-default-efs/IronGear/data/figures/processing/01_processing_summary.png


---
## 11 — Save Processing Report

Saves a comprehensive JSON report to `IronGear/data/reports/processing_report.json` documenting all processing decisions and outputs.


In [13]:
import datetime

# Compile report — cast all numeric values to native Python types for JSON serialisation
processing_report = {
    "generated_at"       : datetime.datetime.now().isoformat(),

    # Dataset summary
    "total_source_images" : int(len(master)),
    "unique_patients"     : int(master["patient_id"].nunique()),

    # Split sizes (images)
    "base_split_images"   : {
        s: int(master[master.split == s].shape[0]) for s in ["train","val","test"]
    },
    "final_split_images"  : {
        "train": int(total_train),
        "val"  : int(total_val),
        "test" : int(total_test),
    },

    # Split sizes (patients)
    "split_patients"      : {
        s: int(master.loc[master.split == s, "patient_id"].nunique())
        for s in ["train","val","test"]
    },

    # Annotation stats
    "annotations_kept"    : int(stats["kept"]),
    "annotations_dropped" : int(stats["dropped"]),
    "negative_images"     : int(stats["negatives"]),
    "per_class_annotations": {k: int(v) for k,v in stats["per_class"].items()},

    # Processing config
    "oversample_config"   : OVERSAMPLE,
    "oversampled_copies_added": {k: int(v) for k,v in os_added.items()},
    "leakage_check"       : "PASSED",
    "seed"                : SEED,

    # Output locations
    "dataset_yaml"        : str(DATASET_YAML),
    "master_index_csv"    : str(master_csv),
}

report_path = REPORTS_DIR / "processing_report.json"
with open(report_path, "w") as f:
    json.dump(processing_report, f, indent=2)

print("=" * 60)
print(" NOTEBOOK 01 COMPLETE — Data Processing")
print("=" * 60)
print(f"\n  YOLO dataset  : {YOLO_DIR}")
print(f"  dataset.yaml  : {DATASET_YAML}")
print(f"  Master index  : {master_csv}")
print(f"  Report        : {report_path}")
print(f"  Figure        : {FIG_DIR / '01_processing_summary.png'}")
print(f"\n  Train images  : {total_train:,} (incl. {total_train - base_train:,} oversampled)")
print(f"  Val images    : {total_val:,}")
print(f"  Test images   : {total_test:,}")
print()
print("  Next step  →  Run: IronGear/Sprint1-POC/notebooks/02_train_evaluate.ipynb")

 NOTEBOOK 01 COMPLETE — Data Processing

  YOLO dataset  : /home/sagemaker-user/user-default-efs/IronGear/data/yolo_dataset
  dataset.yaml  : /home/sagemaker-user/user-default-efs/IronGear/data/yolo_dataset/dataset.yaml
  Master index  : /home/sagemaker-user/user-default-efs/IronGear/data/reports/master_index.csv
  Report        : /home/sagemaker-user/user-default-efs/IronGear/data/reports/processing_report.json
  Figure        : /home/sagemaker-user/user-default-efs/IronGear/data/figures/processing/01_processing_summary.png

  Train images  : 23,454 (incl. 9,211 oversampled)
  Val images    : 2,969
  Test images   : 3,115

  Next step  →  Run: IronGear/Sprint1-POC/notebooks/02_train_evaluate.ipynb
